# 04 — Continuous-time models

Latent ODE and Neural CDE are evaluated on the same hourly PhysioNet 2012 representation used by the baseline models.

The Latent ODE models latent dynamics with an ODE solver and reconstructs observed variables along the trajectory. The Neural CDE instead evolves its hidden state under a continuous control path built from time, measurements and missingness.

Model selection and the Event-1 threshold use a stratified validation split of Set A. Final models are retrained on all Set A before retrospective evaluation on Sets B and C.

In [1]:
import sys, copy, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Start Jupyter from repository root or notebooks/.")

sys.path.insert(0, str(PROJECT_ROOT))

from src.data.physionet import load_split
from src.data.preprocessing import Physionet2012Dataset, build_vocab, fill_forward
from src.models.latent_ode import LatentODE, kl_standard_normal, masked_mse
from src.models.neural_cde import NeuralCDE
from src.training.metrics import select_event1_threshold, challenge_metrics

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def reset_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

reset_seeds()

print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

Project root: C:\Users\guido\git_projects\icu-continuous-forecasting
PyTorch: 2.14.0+cpu
Device: cpu


## Development split

In [2]:
A = load_split(RAW_DIR, split="set-a")
assert len(A) == 4000
assert all(r.label is not None for r in A)

yA = np.asarray([int(r.label) for r in A], dtype=np.int64)
idx = np.arange(len(A))

tr_idx, va_idx = train_test_split(
    idx,
    test_size=0.20,
    random_state=SEED,
    stratify=yA,
)

A_tr = [A[i] for i in tr_idx]
A_va = [A[i] for i in va_idx]

dev_vocab = build_vocab(A_tr)

dev_tr_ds = Physionet2012Dataset(A_tr, dev_vocab)
dev_norm = dev_tr_ds.fit_normalizer()
dev_va_ds = Physionet2012Dataset(A_va, dev_vocab, normalizer=dev_norm)

print("A train/val:", len(A_tr), len(A_va))
print("Vocabulary:", len(dev_vocab))
print("Mortality:", round(yA[tr_idx].mean(), 4), round(yA[va_idx].mean(), 4))

A train/val: 3200 800
Vocabulary: 36
Mortality: 0.1384 0.1388


## Training

In [3]:
BATCH = 128
MAX_EPOCHS = 30
PATIENCE = 6
LR = 1e-3
GRAD_CLIP = 5.0
RECON_WEIGHT = 0.1
KL_WEIGHT = 1e-3

MODEL_CONFIG = {
    "Latent ODE": {
        "kind": "latent_ode",
        "factory": lambda D, S: LatentODE(
            input_dim=D,
            static_dim=S,
            latent_dim=16,
            hidden=64,
            solver="dopri5",
        ),
    },
    "Neural CDE": {
        "kind": "neural_cde",
        "factory": lambda D, S: NeuralCDE(
            input_dim=D,
            static_dim=S,
            hidden_channels=32,
            hidden=64,
            interpolation="cubic",
            solver="rk4",
            step_size=1.0,
        ),
    },
}

def iter_batches(ds, batch_size=BATCH, shuffle=False, epoch=0):
    ids = np.arange(len(ds))

    if shuffle:
        rng = np.random.RandomState(SEED + int(epoch))
        rng.shuffle(ids)

    for start in range(0, len(ids), batch_size):
        yield ds.collate(ids[start:start + batch_size].tolist()).to(DEVICE)

def filled_from_batch(batch):
    x = batch.values.detach().cpu().numpy()
    m = batch.mask.detach().cpu().numpy()
    ff = fill_forward(x, m).astype(np.float32)
    return torch.from_numpy(ff).to(batch.values.device)

def forward_model(kind, model, batch, training):
    if kind == "latent_ode":
        filled = filled_from_batch(batch)
        logits, recon, mean, logvar = model(
            batch,
            filled,
            sample=training,
        )
        return logits, (recon, mean, logvar)

    if kind == "neural_cde":
        return model(batch), None

    raise ValueError(kind)

def train_epoch(kind, model, ds, optimizer, epoch):
    model.train()
    bce_fn = nn.BCEWithLogitsLoss()

    total = 0.0
    n = 0

    for batch in iter_batches(ds, shuffle=True, epoch=epoch):
        optimizer.zero_grad(set_to_none=True)

        logits, aux = forward_model(kind, model, batch, training=True)
        bce = bce_fn(logits, batch.labels.float())
        loss = bce

        if kind == "latent_ode":
            recon, mean, logvar = aux
            rec = masked_mse(recon, batch.values, batch.mask).mean()
            kl = kl_standard_normal(mean, logvar).mean()
            loss = bce + RECON_WEIGHT * rec + KL_WEIGHT * kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        bs = len(batch.labels)
        total += float(loss.detach()) * bs
        n += bs

    return total / n

@torch.no_grad()
def predict(kind, model, ds):
    model.eval()
    out = []

    for batch in iter_batches(ds, shuffle=False):
        logits, _ = forward_model(kind, model, batch, training=False)
        out.append(torch.sigmoid(logits).cpu().numpy())

    return np.concatenate(out)

def labels_numpy(ds):
    return np.asarray(
        [np.nan if r.label is None else float(r.label) for r in ds.records],
        dtype=np.float32,
    )

def develop(name, train_ds, val_ds):
    config = MODEL_CONFIG[name]
    kind = config["kind"]

    D = len(train_ds.vocab)
    S = len(train_ds.static_names)

    reset_seeds()
    model = config["factory"](D, S).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    y_val = labels_numpy(val_ds)
    best_ap = -np.inf
    best_epoch = None
    best_state = None
    stale = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.perf_counter()

        loss = train_epoch(kind, model, train_ds, optimizer, epoch)
        risk = predict(kind, model, val_ds)

        auc = roc_auc_score(y_val, risk)
        ap = average_precision_score(y_val, risk)
        elapsed = time.perf_counter() - t0

        history.append({
            "epoch": epoch,
            "loss": loss,
            "val_AUROC": auc,
            "val_AUPRC": ap,
            "seconds": elapsed,
        })

        if ap > best_ap + 1e-5:
            best_ap = ap
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
            status = "[BEST]"
        else:
            stale += 1
            status = f"[patience {stale}/{PATIENCE}]"

        print(
            f"[{name}] {epoch:02d} loss={loss:.4f} "
            f"val_AUROC={auc:.4f} val_AUPRC={ap:.4f} "
            f"time={elapsed:.1f}s {status}"
        )

        if stale >= PATIENCE:
            break

    model.load_state_dict(best_state)
    risk = predict(kind, model, val_ds)
    threshold = select_event1_threshold(y_val, risk)

    return model, best_epoch, threshold, pd.DataFrame(history)

In [4]:
dev_models = {}
histories = {}
selection_rows = []

for name in MODEL_CONFIG:
    model, epochs, threshold, history = develop(
        name,
        dev_tr_ds,
        dev_va_ds,
    )

    dev_models[name] = model
    histories[name] = history

    best = history.loc[history["epoch"] == epochs].iloc[0]

    selection_rows.append({
        "Model": name,
        "epochs": epochs,
        "threshold": threshold,
        "val_AUROC": best["val_AUROC"],
        "val_AUPRC": best["val_AUPRC"],
    })

selection = pd.DataFrame(selection_rows).set_index("Model")
display(selection.round(4))

[Latent ODE] 01 loss=0.6687 val_AUROC=0.7448 val_AUPRC=0.2987 time=21.2s [BEST]
[Latent ODE] 02 loss=0.4878 val_AUROC=0.7877 val_AUPRC=0.3446 time=23.6s [BEST]
[Latent ODE] 03 loss=0.4585 val_AUROC=0.8008 val_AUPRC=0.3577 time=23.7s [BEST]
[Latent ODE] 04 loss=0.4420 val_AUROC=0.8249 val_AUPRC=0.3981 time=23.4s [BEST]
[Latent ODE] 05 loss=0.4297 val_AUROC=0.8296 val_AUPRC=0.4244 time=23.3s [BEST]
[Latent ODE] 06 loss=0.4178 val_AUROC=0.8402 val_AUPRC=0.4477 time=26.5s [BEST]
[Latent ODE] 07 loss=0.4064 val_AUROC=0.8448 val_AUPRC=0.4787 time=29.2s [BEST]
[Latent ODE] 08 loss=0.3981 val_AUROC=0.8524 val_AUPRC=0.4941 time=27.3s [BEST]
[Latent ODE] 09 loss=0.3867 val_AUROC=0.8576 val_AUPRC=0.5058 time=26.8s [BEST]
[Latent ODE] 10 loss=0.3719 val_AUROC=0.8595 val_AUPRC=0.5012 time=29.7s [patience 1/6]
[Latent ODE] 11 loss=0.3686 val_AUROC=0.8662 val_AUPRC=0.5135 time=34.8s [BEST]
[Latent ODE] 12 loss=0.3588 val_AUROC=0.8582 val_AUPRC=0.5020 time=23.5s [patience 1/6]
[Latent ODE] 13 loss=0.3

,epochs,threshold,val_AUROC,val_AUPRC
Model,,,,
Latent ODE,11,0.2919,0.8662,0.5135
Neural CDE,4,0.2481,0.7814,0.3613


## Final fit

In [5]:
final_vocab = build_vocab(A)

A_ds = Physionet2012Dataset(A, final_vocab)
final_norm = A_ds.fit_normalizer()

assert final_vocab == dev_vocab

D = len(final_vocab)
S = len(A_ds.static_names)

def train_fixed(name, ds, epochs):
    config = MODEL_CONFIG[name]
    kind = config["kind"]

    reset_seeds()
    model = config["factory"](D, S).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, int(epochs) + 1):
        t0 = time.perf_counter()
        loss = train_epoch(kind, model, ds, optimizer, epoch)

        print(
            f"[{name}] {epoch:02d}/{int(epochs):02d} "
            f"loss={loss:.4f} time={time.perf_counter() - t0:.1f}s"
        )

    return model

final_models = {
    name: train_fixed(name, A_ds, selection.loc[name, "epochs"])
    for name in MODEL_CONFIG
}

[Latent ODE] 01/11 loss=0.6396 time=22.0s
[Latent ODE] 02/11 loss=0.4724 time=23.5s
[Latent ODE] 03/11 loss=0.4421 time=23.6s
[Latent ODE] 04/11 loss=0.4284 time=23.7s
[Latent ODE] 05/11 loss=0.4150 time=23.5s
[Latent ODE] 06/11 loss=0.4054 time=22.6s
[Latent ODE] 07/11 loss=0.3920 time=22.4s
[Latent ODE] 08/11 loss=0.3862 time=22.7s
[Latent ODE] 09/11 loss=0.3741 time=22.7s
[Latent ODE] 10/11 loss=0.3660 time=22.6s
[Latent ODE] 11/11 loss=0.3534 time=22.7s
[Neural CDE] 01/04 loss=0.4653 time=388.8s
[Neural CDE] 02/04 loss=0.3576 time=378.0s
[Neural CDE] 03/04 loss=0.3362 time=380.4s
[Neural CDE] 04/04 loss=0.3323 time=376.3s


## Retrospective evaluation

In [6]:
B = load_split(RAW_DIR, split="set-b")
C = load_split(RAW_DIR, split="set-c")

assert len(B) == len(C) == 4000
assert all(r.label is None for r in B)
assert all(r.label is None for r in C)

B_ds = Physionet2012Dataset(B, final_vocab, normalizer=final_norm)
C_ds = Physionet2012Dataset(C, final_vocab, normalizer=final_norm)

risk_B = {}
risk_C = {}

for name, config in MODEL_CONFIG.items():
    kind = config["kind"]
    risk_B[name] = predict(kind, final_models[name], B_ds)
    risk_C[name] = predict(kind, final_models[name], C_ds)

print("PREDICTIONS FROZEN. B/C outcomes have not been read up to this point.")

PREDICTIONS FROZEN. B/C outcomes have not been read up to this point.


In [7]:
def read_outcomes(path):
    df = pd.read_csv(path, dtype={"RecordID": str})
    df["RecordID"] = df["RecordID"].str.strip()
    assert df["RecordID"].is_unique
    return df.set_index("RecordID")["In-hospital_death"]

def aligned_y(records, outcome_map):
    ids = [str(r.record_id).strip() for r in records]

    missing = sorted(set(ids) - set(outcome_map.index))
    extra = sorted(set(outcome_map.index) - set(ids))

    assert not missing, f"Missing outcome IDs: {missing[:10]}"
    assert not extra, f"Outcome IDs without records: {extra[:10]}"

    return np.asarray(
        [int(outcome_map.loc[rid]) for rid in ids],
        dtype=np.int64,
    )

yB = aligned_y(B, read_outcomes(RAW_DIR / "Outcomes-b.txt"))
yC = aligned_y(C, read_outcomes(RAW_DIR / "Outcomes-c.txt"))

rows = []

for name in MODEL_CONFIG:
    threshold = float(selection.loc[name, "threshold"])

    rows.append({
        "Model": name,
        "Set": "B",
        **challenge_metrics(yB, risk_B[name], threshold),
    })
    rows.append({
        "Model": name,
        "Set": "C",
        **challenge_metrics(yC, risk_C[name], threshold),
    })

results = pd.DataFrame(rows).rename(columns={
    "auroc": "AUROC",
    "auprc": "AUPRC",
    "event1": "Event1",
    "sensitivity": "Sensitivity",
    "ppv": "PPV",
    "threshold": "Threshold",
    "event2": "Event2",
})

display(results.round(4))

,Model,Set,AUROC,AUPRC,Event1,Sensitivity,PPV,Threshold,Event2
0,Latent ODE,B,0.8451,0.4780,0.4120,0.4120,0.5200,0.2919,267.6598
1,Latent ODE,C,0.8451,0.5125,0.4120,0.4120,0.5391,0.2919,358.9472
2,Neural CDE,B,0.7523,0.3197,0.3597,0.3768,0.3597,0.2481,50.1927
3,Neural CDE,C,0.7516,0.3448,0.3625,0.3829,0.3625,0.2481,122.0344


## Summary

The hourly representation separates the effect of continuous latent dynamics from the effect of preserving raw observation times. The Latent ODE and Neural CDE are therefore compared here as modeling choices under a common discretized input representation; the exact-time ablation is evaluated separately.

In [8]:
c_results = (
    results[results["Set"] == "C"]
    .sort_values("AUROC", ascending=False)
    .reset_index(drop=True)
)

display(c_results.round(4))

assert len(results) == 4
assert set(results["Set"]) == {"B", "C"}
assert np.isfinite(
    results[["AUROC", "AUPRC", "Event1", "Event2"]].to_numpy()
).all()

print("Notebook 04 completed successfully.")

,Model,Set,AUROC,AUPRC,Event1,Sensitivity,PPV,Threshold,Event2
0,Latent ODE,C,0.8451,0.5125,0.4120,0.4120,0.5391,0.2919,358.9472
1,Neural CDE,C,0.7516,0.3448,0.3625,0.3829,0.3625,0.2481,122.0344


Notebook 04 completed successfully.
